# OM 620: Safety Stock Analysis (Assignments 1 and 2)

**Group:** Daniel Ramirez, Diego Holguin, Jorge Hernandez Ramirez, Suleiman Sahawneh

**Course:** OM 620 Tools and Technologies for Analytics, CSUSM, Fall 2026

## What we are doing

A company keeps extra items on the shelf so it does not run out when orders come in. This extra amount is called **safety stock**.

In this notebook we take the company's order data, clean it, and then calculate how much safety stock each product needs.

The steps are:

1. Load the data
2. Fix the column names
3. Find and fix bad numbers
4. Handle missing values
5. Add a country column
6. Keep only the products that need safety stock
7. Answer some basic questions about those products
8. Summarize each product
9. Calculate safety stock
10. Look at the results
11. Try a second method (side quest)
12. Wrap up

## Setup

We load the tools we need:

- `pandas` for working with tables of data
- `numpy` for the country step
- `norm` from SciPy to get the service level numbers
- `mquantiles` from SciPy for the side quest

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.stats.mstats import mquantiles

## Step 1: Load the data

We have two files in the `data` folder:

- `transaction_data.csv` has the orders. Each row is one order.
- `meta_data.csv` explains what each column means.

In [2]:
transactions = pd.read_csv("../data/transaction_data.csv")
metadata = pd.read_csv("../data/meta_data.csv")
transactions.head()

,sku_numb3r,inventory_type,stocking_type,lead_time,unit_price,Manufacturing Site,division_code,transaction_date,Order-Quantity
0,C4CCB7BB,FG,MTS,28,1014.553490,US-1F,AF8D,2023-01-01,50
1,1CC9F4D5,FG,MTS,28,1014.698985,NaN,A873,2023-01-01,105
2,88D82D16,WIP,MTS,28,1003.032674,PL-FA,8CF4,2023-01-01,105
3,5DDF2B93,FG,MTO,28,1009.165291,CA-D0,1ED9,2023-01-01,129
4,20A7E2D1,RM,MTS,28,1007.309891,US-F1,ECB9,2023-01-01,133


In [3]:
transactions.shape

(728463, 9)

The order file has 728,463 rows and 9 columns.

## Step 2: Fix the column names

The column names in the order file should match the names in the metadata file. Let's compare them.

In [4]:
metadata

,feature,description
0,sku_number,Eight character stock keeping unit number
1,inventory_type,Types of inventory including finished good - w...
2,stocking_type,Stockign type policy including make to stock -...
3,lead_time,Total lead time in days
4,unit_price,Unit price US dollar value
5,manufacturing_site,Manufacturing site ID starting with country co...
6,division_code,Four character business division ID
7,transaction_date,Date of Transaction in Year-Month-Day format
8,order_quantity,Order quantity - rounded up for convenience


In [5]:
transactions.columns

Index(['sku_numb3r', 'inventory_type', 'stocking_type', 'lead_time',
       'unit_price', 'Manufacturing Site', 'division_code', 'transaction_date',
       'Order-Quantity'],
      dtype='str')

Three column names do not match:

- `sku_numb3r` has a 3 instead of an e. It should be `sku_number`.
- `Manufacturing Site` has capitals and a space. It should be `manufacturing_site`.
- `Order-Quantity` has capitals and a dash. It should be `order_quantity`.

We copy each one into a column with the correct name, then keep the nine correct columns in order.

In [6]:
transactions["sku_number"] = transactions["sku_numb3r"]
transactions["manufacturing_site"] = transactions["Manufacturing Site"]
transactions["order_quantity"] = transactions["Order-Quantity"]
transactions = transactions.reindex(columns=["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price",
                                             "manufacturing_site", "division_code", "transaction_date", "order_quantity"])
transactions.columns

Index(['sku_number', 'inventory_type', 'stocking_type', 'lead_time',
       'unit_price', 'manufacturing_site', 'division_code', 'transaction_date',
       'order_quantity'],
      dtype='str')

All nine column names now match the metadata.

## Step 3: Find and fix bad numbers

`describe()` gives a quick summary of the number columns. We look for values that do not make sense.

In [7]:
transactions.describe()

,lead_time,unit_price,order_quantity
count,728463.000000,728463.000000,728463.000000
mean,25.782708,800.030488,79.899502
std,10.536974,374.802172,41.920227
min,14.000000,-1012.695895,-126.000000
25%,28.000000,980.472511,64.000000
50%,28.000000,997.457813,93.000000
75%,28.000000,1005.804896,109.000000
max,2800.000000,1026.431584,208.000000


Three things look wrong:

- `lead_time` goes up to 2,800 days. Normal lead times are 14, 21 or 28 days.
- `unit_price` goes below zero. A price cannot be negative.
- `order_quantity` goes below zero.

We check and fix each one below.

### Problem 1: Lead times that are too long

In [8]:
transactions["lead_time"].value_counts()

lead_time
28      583162
14       89014
21       56277
2800         7
2100         2
1400         1
Name: count, dtype: int64

10 orders have lead times of 1,400, 2,100 or 2,800 days. Each one is exactly 100 times a normal lead time, so two extra zeros were most likely typed by mistake. We change them back to 14, 21 and 28.

In [9]:
transactions["lead_time"] = transactions["lead_time"].replace({1400: 14, 2100: 21, 2800: 28})
transactions["lead_time"].value_counts()

lead_time
28    583169
14     89015
21     56279
Name: count, dtype: int64

### Problem 2: Negative prices

In [10]:
transactions[transactions["unit_price"] < 0]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
70371,F6AC7C21,WIP,MTS,28,-991.288006,US-F1,5D76,2023-04-16,127
76230,4468BB27,FG,MTS,28,-990.568788,PL-FA,62D5,2023-04-25,105
135594,6625477B,WIP,MTS,28,-993.049890,PL-FA,A873,2023-07-23,114
280662,C5D9A49C,WIP,MTO,28,-990.700010,PL-FA,6B4E,2024-02-26,78
393851,1B589A55,FG,MTO,28,-1012.695895,JP-C7,7164,2024-08-15,86
483775,3408C989,FG,MTO,21,-215.230508,US-1F,E735,2024-12-29,-6
508929,706929BB,RM,MTS,21,-188.490312,JP-C7,E735,2025-02-04,55
573230,74684AEA,FG,MTS,28,-1011.897849,JP-C7,762C,2025-05-11,83
671107,3D4ADA3C,FG,MTS,28,-996.250336,PL-FA,C573,2025-10-05,91
713452,4BFB0581,RM,MTS,28,-999.692246,NaN,91A9,2025-12-08,91


Let's look at one of these products to see its other prices.

In [11]:
transactions[transactions["sku_number"] == "4BFB0581"]["unit_price"].value_counts()

unit_price
 999.692246     1755
 1001.895816    1692
 64.164938        86
-999.692246        1
Name: count, dtype: int64

This product has a price of 999.69 in 1,755 orders and -999.69 in just one. Only the minus sign is wrong, so we remove it with `abs()`.

In [12]:
transactions["unit_price"] = transactions["unit_price"].abs()
(transactions["unit_price"] < 0).sum()

np.int64(0)

### Problem 3: Negative order quantities

In [13]:
(transactions["order_quantity"] < 0).sum(), (transactions["order_quantity"] == 0).sum()

(np.int64(37557), np.int64(4222))

37,557 orders have a negative quantity and 4,222 have a quantity of zero.

At first we thought the negative ones were returns. Our professor's feedback said they are most likely sign mistakes, just like the prices. So we remove the minus sign here too.

We keep the zero orders for now. We check later whether they change our answers.

In [14]:
transactions["order_quantity"] = transactions["order_quantity"].abs()
(transactions["order_quantity"] < 0).sum()

np.int64(0)

### Problem 4: Duplicate rows

In [15]:
transactions.duplicated().sum()

np.int64(47)

Some rows are exact copies of another row. There is no order ID in the data, and a product is often ordered more than once on the same day. These could be real orders, so we keep them.

### The numbers after the fixes

In [16]:
transactions.describe()

,lead_time,unit_price,order_quantity
count,728463.000000,728463.000000,728463.000000
mean,25.748461,800.053522,80.693939
std,4.760940,374.753000,40.369839
min,14.000000,35.786593,0.000000
25%,28.000000,980.472511,64.000000
50%,28.000000,997.457813,93.000000
75%,28.000000,1005.804896,109.000000
max,28.000000,1026.431584,208.000000


Lead times are now 14 to 28 days, all prices are positive, and no order quantity is below zero.

## Step 4: Handle missing values

Now we count the blank cells in each column.

In [17]:
transactions.isna().sum()

sku_number              4109
inventory_type         14631
stocking_type           6923
lead_time                  0
unit_price                 0
manufacturing_site    141071
division_code          50642
transaction_date           0
order_quantity             0
dtype: int64

Five columns have blanks. Here is what we do with them:

- **sku_number:** We delete these rows. We calculate safety stock for each product, and without a product number we cannot tell which product the order belongs to.
- **inventory_type and stocking_type:** We fill these with "Unknown". We cannot guess them from the product, because one product can have more than one type.
- **manufacturing_site:** We fill these with "Unknown". About 1 in 5 rows is blank here, and deleting them would lose too much data.
- **division_code:** We fill these with "Unknown". We do not need it to identify a product.

In [18]:
transactions = transactions[transactions["sku_number"].notna()]
transactions = transactions.fillna({"inventory_type": "Unknown", "stocking_type": "Unknown",
                                    "manufacturing_site": "Unknown", "division_code": "Unknown"})
transactions.isna().sum()

sku_number            0
inventory_type        0
stocking_type         0
lead_time             0
unit_price            0
manufacturing_site    0
division_code         0
transaction_date      0
order_quantity        0
dtype: int64

In [19]:
transactions.shape

(724354, 9)

No blanks are left. We removed 4,109 rows without a product number, which leaves 724,354 rows.

## Step 5: Add a country column

The metadata says each manufacturing site starts with a country code. Let's look at the site codes.

In [20]:
transactions["manufacturing_site"].value_counts()

manufacturing_site
US-1F      140676
PL-FA      140546
Unknown    140293
JP-C7      140231
US-F1       28419
CA-3F       28357
CA-D0       14067
JP-82       14034
US-A2        7162
CA-D4        7128
PL-D7        7110
JP-6D        7102
CA-FB        7100
CA-6A        7088
CA-24        7084
CA-95        7039
JP-BE        7016
US-4E        6990
PL-51        6912
Name: count, dtype: int64

Every site starts with `US-`, `CA-`, `PL-` or `JP-`. We use `str.startswith()` to turn these into a new `country` column. Anything else becomes "Unknown".

In [21]:
site = transactions["manufacturing_site"]
transactions["country"] = np.where(site.str.startswith("US-"), "USA",
                          np.where(site.str.startswith("CA-"), "Canada",
                          np.where(site.str.startswith("PL-"), "Poland",
                          np.where(site.str.startswith("JP-"), "Japan", "Unknown"))))
transactions["country"].value_counts()

country
USA        183247
Japan      168383
Poland     154568
Unknown    140293
Canada      77863
Name: count, dtype: int64

The USA has the most orders, followed by Japan, Poland and Canada. The "Unknown" rows are the sites we filled in Step 4.

## Step 6: Keep only the products that need safety stock

Safety stock is for items we keep on the shelf before a customer orders them. So we keep only rows that are:

- **FG**: finished goods
- **MTS**: made to stock

In [22]:
mts = transactions[(transactions["inventory_type"] == "FG") & (transactions["stocking_type"] == "MTS")]
mts.shape

(267212, 10)

267,212 orders are finished goods made to stock. We use this smaller table, called `mts`, for the rest of the notebook.

## Step 7: Basic questions about these products

### How many different products (SKUs)?

In [23]:
mts["sku_number"].nunique()

375

There are **375** different products.

### How many manufacturing sites?

We leave out "Unknown" because it is not a real site.

In [24]:
mts[mts["manufacturing_site"] != "Unknown"]["manufacturing_site"].nunique()

18

There are **18** sites.

### How many divisions?

In [25]:
mts[mts["division_code"] != "Unknown"]["division_code"].nunique()

54

There are **54** divisions.

### Top 10 and bottom 10 orders by quantity

In [26]:
mts.sort_values(["order_quantity", "unit_price", "transaction_date"], ascending=False).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
476865,A4DFE12D,FG,MTS,28,1009.801193,US-1F,AF8D,2024-12-18,208,USA
603538,9BD808D9,FG,MTS,28,1012.729872,PL-FA,8CF4,2025-06-26,187,Poland
367041,61A9013F,FG,MTS,28,1003.137627,US-1F,62D5,2024-07-06,186,USA
126203,77074CBD,FG,MTS,28,982.747333,JP-C7,62D5,2023-07-09,186,Japan
236313,A963AFB6,FG,MTS,28,997.839409,US-1F,7EA2,2023-12-22,185,USA
423824,0027AFDA,FG,MTS,28,989.192167,JP-C7,Unknown,2024-09-29,183,Japan
120534,13383A60,FG,MTS,28,996.977138,JP-C7,8CF4,2023-06-30,182,Japan
27638,1CC9F4D5,FG,MTS,28,1014.698985,CA-24,ECB9,2023-02-11,181,Canada
680072,74446B56,FG,MTS,28,998.084102,JP-C7,A873,2025-10-19,181,Japan
303661,96BBC0F5,FG,MTS,28,1010.990593,US-F1,2875,2024-04-01,180,USA


In [27]:
mts.sort_values(["order_quantity", "unit_price", "transaction_date"]).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
63628,6B506B1D,FG,MTS,14,35.895733,JP-C7,E122,2023-04-06,0,Japan
304947,6B506B1D,FG,MTS,14,35.895733,US-1F,E735,2024-04-03,0,USA
542891,6B506B1D,FG,MTS,14,35.895733,CA-3F,EF3F,2025-03-27,0,Canada
148508,8996984B,FG,MTS,14,36.059805,JP-C7,8CF4,2023-08-12,0,Japan
419279,8996984B,FG,MTS,14,36.059805,PL-FA,1ED9,2024-09-23,0,Poland
439398,8996984B,FG,MTS,14,36.059805,CA-6A,5D76,2024-10-22,0,Canada
688808,8996984B,FG,MTS,14,36.059805,US-1F,ECB9,2025-11-01,0,USA
209285,B9331D8D,FG,MTS,14,36.078593,US-4E,A873,2023-11-11,0,USA
381570,B9331D8D,FG,MTS,14,36.078593,PL-FA,1F6F,2024-07-28,0,Poland
104208,706929BB,FG,MTS,14,36.725023,Unknown,2875,2023-06-05,0,Unknown


In [28]:
(mts["order_quantity"] == 0).sum()

np.int64(1445)

The largest order is **208 units** (product A4DFE12D). The smallest orders are **0 units**. There are 1,445 zero orders in this table, so the bottom 10 are all zeros.

### Top 10 and bottom 10 orders by sales

The data has no sales column, so we make one: sales = unit price × order quantity.

In [29]:
mts = mts.assign(sales=mts["unit_price"] * mts["order_quantity"])
mts.sort_values(["sales", "transaction_date"], ascending=False).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country,sales
476865,A4DFE12D,FG,MTS,28,1009.801193,US-1F,AF8D,2024-12-18,208,USA,210038.648200
603538,9BD808D9,FG,MTS,28,1012.729872,PL-FA,8CF4,2025-06-26,187,Poland,189380.486099
367041,61A9013F,FG,MTS,28,1003.137627,US-1F,62D5,2024-07-06,186,USA,186583.598655
236313,A963AFB6,FG,MTS,28,997.839409,US-1F,7EA2,2023-12-22,185,USA,184600.290718
27638,1CC9F4D5,FG,MTS,28,1014.698985,CA-24,ECB9,2023-02-11,181,Canada,183660.516368
126203,77074CBD,FG,MTS,28,982.747333,JP-C7,62D5,2023-07-09,186,Japan,182791.004027
303661,96BBC0F5,FG,MTS,28,1010.990593,US-F1,2875,2024-04-01,180,USA,181978.306814
120534,13383A60,FG,MTS,28,996.977138,JP-C7,8CF4,2023-06-30,182,Japan,181449.839088
448644,1DF2F967,FG,MTS,28,1006.255871,PL-FA,2875,2024-11-05,180,Poland,181126.056746
423824,0027AFDA,FG,MTS,28,989.192167,JP-C7,Unknown,2024-09-29,183,Japan,181022.166518


In [30]:
mts.sort_values(["sales", "transaction_date"]).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country,sales
137,6E4AD684,FG,MTS,14,53.481465,CA-D4,62D5,2023-01-01,0,Canada,0.0
552,F1FA52E2,FG,MTS,21,194.501553,PL-D7,A9D5,2023-01-01,0,Poland,0.0
597,875EC717,FG,MTS,21,186.474407,JP-C7,2581,2023-01-01,0,Japan,0.0
1313,EB75221B,FG,MTS,14,46.716458,US-1F,8E14,2023-01-02,0,USA,0.0
2958,BF4F3672,FG,MTS,14,45.311821,US-1F,2875,2023-01-05,0,USA,0.0
3589,2E65B615,FG,MTS,14,47.748103,PL-FA,8CF4,2023-01-06,0,Poland,0.0
4048,0D8F9792,FG,MTS,14,49.562872,Unknown,2875,2023-01-07,0,Unknown,0.0
4325,1F050974,FG,MTS,21,212.412035,JP-82,0C30,2023-01-07,0,Japan,0.0
4452,C6A7BBE5,FG,MTS,14,44.942937,JP-C7,A873,2023-01-07,0,Japan,0.0
5685,FC9F2FBB,FG,MTS,14,53.985175,Unknown,A873,2023-01-09,0,Unknown,0.0


The largest sale is **$210,038.65**. It is the same order as the largest quantity. The smallest sales are **$0**, which are the zero orders.

## Step 8: Summarize each product

To calculate safety stock, we need a few numbers for each product. We group the orders by product and find:

- the smallest, largest, average and middle (median) order
- the variance and standard deviation, which show how much the orders jump around
- the average lead time

In [31]:
sku_stats = (
    mts
    .groupby("sku_number")
    .agg(
        min_quantity=("order_quantity", "min"),
        max_quantity=("order_quantity", "max"),
        mean_quantity=("order_quantity", "mean"),
        median_quantity=("order_quantity", "median"),
        var_quantity=("order_quantity", "var"),
        std_quantity=("order_quantity", "std"),
        mean_lead_time=("lead_time", "mean"),
    )
)
sku_stats.round(2)

,min_quantity,max_quantity,mean_quantity,median_quantity,var_quantity,std_quantity,mean_lead_time
sku_number,,,,,,,
0027AFDA,39,183,100.55,101.0,398.99,19.97,28.00
01637C60,0,35,8.04,7.0,47.17,6.87,14.00
02F3C196,0,30,8.39,7.0,37.39,6.11,14.00
04711B22,0,25,8.24,7.0,33.73,5.81,14.00
05B8D97A,23,160,100.42,100.0,374.61,19.35,28.00
...,...,...,...,...,...,...,...
FD350FE2,0,169,87.33,96.0,1340.90,36.62,26.04
FD3CB2F5,13,161,99.48,100.0,418.05,20.45,28.00
FDC306C8,1,29,8.94,8.0,41.08,6.41,14.00


We now have one row for each of the 375 products.

Let's see if the products fall into groups. We split them by standard deviation at 15 units.

In [32]:
sku_stats.groupby(sku_stats["std_quantity"] > 15).agg(
    skus=("std_quantity", "size"),
    average_quantity=("mean_quantity", "mean"),
    average_lead_time=("mean_lead_time", "mean"),
).round(2)

,skus,average_quantity,average_lead_time
std_quantity,,,
False,225,9.45,15.29
True,150,77.08,25.75


There are two clear groups:

- **225 products** with small orders (about 9 units) and shorter lead times (about 15 days)
- **150 products** with big orders (about 77 units) and longer lead times (about 26 days)

In [33]:
lead_check = mts.groupby("sku_number").agg(
    lowest_lead_time=("lead_time", "min"),
    highest_lead_time=("lead_time", "max"),
)
(lead_check["lowest_lead_time"] != lead_check["highest_lead_time"]).sum()

np.int64(87)

87 products were shipped with more than one lead time. That is why some average lead times are not whole numbers.

## Step 9: Calculate safety stock

We use the formula from the assignment:

**Safety stock = z × standard deviation × square root of lead time**

- **z** is the service level number. A higher service level means we run out less often, so we need more stock.
- **standard deviation** is how much the product's orders jump around (from Step 8).
- **lead time** is the product's average lead time in days (from Step 8).

First we get z for 75%, 90% and 95%.

In [34]:
z_75 = norm.ppf(0.75)
z_90 = norm.ppf(0.90)
z_95 = norm.ppf(0.95)
z_75, z_90, z_95

(np.float64(0.6744897501960817), np.float64(1.2815515655446004), np.float64(1.6448536269514722))

The z values are 0.674 for 75%, 1.282 for 90% and 1.645 for 95%. Now we calculate safety stock for every product at each level.

In [35]:
safety_stock = sku_stats.assign(
    ss_75=z_75 * sku_stats["std_quantity"] * sku_stats["mean_lead_time"] ** 0.5,
    ss_90=z_90 * sku_stats["std_quantity"] * sku_stats["mean_lead_time"] ** 0.5,
    ss_95=z_95 * sku_stats["std_quantity"] * sku_stats["mean_lead_time"] ** 0.5,
)
safety_stock[["std_quantity", "mean_lead_time", "ss_75", "ss_90", "ss_95"]].round(2)

,std_quantity,mean_lead_time,ss_75,ss_90,ss_95
sku_number,,,,,
0027AFDA,19.97,28.00,71.29,135.46,173.86
01637C60,6.87,14.00,17.33,32.93,42.27
02F3C196,6.11,14.00,15.43,29.32,37.63
04711B22,5.81,14.00,14.66,27.85,35.74
05B8D97A,19.35,28.00,69.08,131.25,168.46
...,...,...,...,...,...
FD350FE2,36.62,26.04,126.04,239.48,307.37
FD3CB2F5,20.45,28.00,72.97,138.65,177.96
FDC306C8,6.41,14.00,16.18,30.73,39.45


In [36]:
safety_stock[["ss_75", "ss_90", "ss_95"]].mean().round(2)

ss_75     43.64
ss_90     82.92
ss_95    106.42
dtype: float64

The average safety stock per product goes up as the service level goes up: about 44 units at 75%, 83 units at 90% and 106 units at 95%.

## Step 10: Look at the results (95% service level)

### Largest safety stock

In [37]:
safety_stock.sort_values("ss_95", ascending=False).head(1)[["mean_quantity", "std_quantity", "mean_lead_time", "ss_95"]]

,mean_quantity,std_quantity,mean_lead_time,ss_95
sku_number,,,,
084B19C4,75.108714,42.143834,25.952282,353.14176


### Smallest safety stock

In [38]:
safety_stock.sort_values("ss_95").head(1)[["mean_quantity", "std_quantity", "mean_lead_time", "ss_95"]]

,mean_quantity,std_quantity,mean_lead_time,ss_95
sku_number,,,,
FE8A9BBE,7.542169,5.351775,14.0,32.937384


### Average safety stock

In [39]:
safety_stock["ss_95"].mean()

np.float64(106.42146831675336)

What we found:

- **Largest:** product 084B19C4 needs about 353 units. Its orders change the most of any product.
- **Smallest:** product FE8A9BBE needs about 33 units. Its orders are steady and its lead time is short.
- **Average:** about 106 units per product.

The largest is about 11 times the smallest. Giving every product the same amount would not work. Some would run out and others would have too much stock.

### Do the zero orders change the answer?

A zero order may not be real demand. We recalculate the 95% safety stock without the zero orders and compare.

In [40]:
orders_only = mts[mts["order_quantity"] > 0].groupby("sku_number").agg(
    std_quantity=("order_quantity", "std"),
    mean_lead_time=("lead_time", "mean"),
)
orders_only = orders_only.assign(ss_95=z_95 * orders_only["std_quantity"] * orders_only["mean_lead_time"] ** 0.5)

pd.DataFrame(
    {
        "with_zeros": [safety_stock["ss_95"].max(), safety_stock["ss_95"].min(), safety_stock["ss_95"].mean()],
        "without_zeros": [orders_only["ss_95"].max(), orders_only["ss_95"].min(), orders_only["ss_95"].mean()],
    },
    index=["largest", "smallest", "average"],
).round(2)

,with_zeros,without_zeros
largest,353.14,350.63
smallest,32.94,31.72
average,106.42,105.58


Removing the zeros changes the average by less than one unit, and the largest and smallest products stay the same. So we keep the zero orders.

## Step 11: Side quest, a second method

The formula in Step 9 assumes each product's orders follow a bell curve. Not every product does. For example, a product shipped with different lead times may have different order sizes for each one.

So we try a second method that uses each product's real order history instead:

**Safety stock = (95th percentile order − average order) × square root of lead time**

`mquantiles` finds the 75th, 90th and 95th percentile order for each product.

In [41]:
quantiles = mts.groupby("sku_number").agg(
    q_75=("order_quantity", lambda x: mquantiles(x, prob=[0.75])[0]),
    q_90=("order_quantity", lambda x: mquantiles(x, prob=[0.90])[0]),
    q_95=("order_quantity", lambda x: mquantiles(x, prob=[0.95])[0]),
)
empirical = sku_stats.assign(q_75=quantiles["q_75"], q_90=quantiles["q_90"], q_95=quantiles["q_95"])
empirical = empirical.assign(
    ss_emp_75=(empirical["q_75"] - empirical["mean_quantity"]) * empirical["mean_lead_time"] ** 0.5,
    ss_emp_90=(empirical["q_90"] - empirical["mean_quantity"]) * empirical["mean_lead_time"] ** 0.5,
    ss_emp_95=(empirical["q_95"] - empirical["mean_quantity"]) * empirical["mean_lead_time"] ** 0.5,
)
empirical[["mean_quantity", "q_95", "ss_emp_95"]].round(2)

,mean_quantity,q_95,ss_emp_95
sku_number,,,
0027AFDA,100.55,133.00,171.69
01637C60,8.04,22.17,52.89
02F3C196,8.39,19.94,43.23
04711B22,8.24,18.68,39.06
05B8D97A,100.42,133.00,172.39
...,...,...,...
FD350FE2,87.33,131.00,222.83
FD3CB2F5,99.48,133.00,177.39
FDC306C8,8.94,21.00,45.14


### Comparing the two methods

In [42]:
pd.DataFrame(
    {
        "formula_average": [safety_stock["ss_75"].mean(), safety_stock["ss_90"].mean(), safety_stock["ss_95"].mean()],
        "real_history_average": [empirical["ss_emp_75"].mean(), empirical["ss_emp_90"].mean(), empirical["ss_emp_95"].mean()],
    },
    index=["75%", "90%", "95%"],
).round(2)

,formula_average,real_history_average
75%,43.64,41.32
90%,82.92,79.78
95%,106.42,103.06


In [43]:
comparison = safety_stock[["ss_95"]].assign(ss_emp_95=empirical["ss_emp_95"])
comparison.loc[["084B19C4", "FE8A9BBE", "A6AB6A6D"]].round(2)

,ss_95,ss_emp_95
sku_number,,
084B19C4,353.14,279.63
FE8A9BBE,32.94,37.03
A6AB6A6D,35.74,33.53


Let's see why product 084B19C4 changes so much. We look at its orders by lead time.

In [44]:
mts[mts["sku_number"] == "084B19C4"].groupby("lead_time").agg(
    orders=("order_quantity", "size"),
    average_quantity=("order_quantity", "mean"),
).round(2)

,orders,average_quantity
lead_time,,
14,183,9.72
21,339,23.12
28,1888,90.78


What we found:

- The real-history method gives a slightly lower average: about 103 units at 95%, compared with 106 from the formula.
- Product 084B19C4 is still the largest, but it drops from about 353 to about 280 units. Its orders are small when shipped in 14 days and large when shipped in 28 days. Mixing these together makes the formula think its orders jump around more than they really do.
- The smallest product is now A6AB6A6D, at about 34 units.

So the formula can overestimate safety stock for products whose orders do not follow a bell curve.

## Step 12: Wrap up

We started with 728,463 orders that had wrong column names, bad numbers and missing values. Here is what we did:

1. Fixed three column names.
2. Fixed lead times that were 100 times too big.
3. Removed the minus sign from negative prices and order quantities.
4. Deleted rows with no product number and filled other blanks with "Unknown".
5. Added a country column.
6. Kept only finished goods made to stock: 267,212 orders and 375 products.
7. Summarized each product and calculated safety stock.

**Final result at a 95% service level:**

- Average safety stock is about **106 units** per product.
- It ranges from about **33 units** to about **353 units**, depending on how much each product's orders change.
- A second method using real order history gives a similar average of about **103 units**.

## References

- OM 620 course lectures, notebooks and assignment instructions (CSUSM, Fall 2026)
- Our professor's feedback on Assignment 1
- SciPy documentation for `norm` and `mquantiles`: https://docs.scipy.org